## Middleware

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Built-In Middleware : Summarization Middleware 

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool

#### Message Based Summarization

In [ ]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",  
    checkpointer=InMemorySaver(),  
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",  
            trigger=("messages", 10),  
            keep=("messages", 4)  
        ),  
    ]
)


In [ ]:
# Run with a thread
config={"configurable": {"thread_id": "test-1"}}

# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

#### Token Based Summarization

In [ ]:
# Tool Example
@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="groq:openai/gpt-oss-120b",  
    checkpointer=InMemorySaver(), 
    tools=[search_hotels], 
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",  
            trigger=("tokens", 550),  
            keep=("tokens", 200)  
        ),  
    ]
)

# Run with a thread
config={"configurable": {"thread_id": "test-1"}}

# Token Counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars = 1 token

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

#### Context Fraction Based Summarization

In [ ]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("fraction", 0.005),  # 0.5% = ~655 tokens
            keep=("fraction", 0.002),     # 0.2% = ~262 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 131072   # gpt-oss-120b context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages']) 